# Tarea 1 — Limpieza de datos: concreto

**Objetivo.** Revisar la calidad de `Concrete_Data.csv`, estandarizar los nombres y tipos, detectar duplicados y marcar valores atípicos con IQR. Se conserva la base original; solo se eliminan duplicados exactos si los hubiera.

La base tiene variables de composición de la mezcla y edad del concreto. La respuesta `resistencia_mpa` mide resistencia a la compresión en MPa.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

candidatos = [Path("Concrete_Data.csv"), Path("../../Concrete_Data.csv"), Path("../Concrete_Data.csv")]
archivo = next((p for p in candidatos if p.exists()), None)
if archivo is None:
    raise FileNotFoundError("No se encontró Concrete_Data.csv; abre el notebook desde una carpeta del proyecto.")

nombres = ["cemento", "escoria", "ceniza_volante", "agua", "superplastificante",
           "agregado_grueso", "agregado_fino", "edad_dias", "resistencia_mpa"]
datos = pd.read_csv(archivo)
if datos.shape[1] != len(nombres):
    raise ValueError("Se esperaban 9 columnas en Concrete_Data.csv")
datos.columns = nombres
datos = datos.apply(pd.to_numeric, errors="coerce")
print(f"Dimensiones: {datos.shape[0]} filas × {datos.shape[1]} columnas")
datos.head()

Dimensiones: 1030 filas × 9 columnas


,cemento,escoria,ceniza_volante,agua,superplastificante,agregado_grueso,agregado_fino,edad_dias,resistencia_mpa
0,540.0,0.0,0.0,162.0,2.5,1040.0,676.0,28,79.986111
1,540.0,0.0,0.0,162.0,2.5,1055.0,676.0,28,61.887366
2,332.5,142.5,0.0,228.0,0.0,932.0,594.0,270,40.269535
3,332.5,142.5,0.0,228.0,0.0,932.0,594.0,365,41.052780
4,198.6,132.4,0.0,192.0,0.0,978.4,825.5,360,44.296075


## Diagnóstico inicial

Cuantifica faltantes y duplicados y revisa rangos. Si aparecen extremos, se investigan antes de decidir si deben modificarse: las proporciones de la mezcla y la edad pueden generar observaciones extremas válidas.

In [2]:
print("Tipos de datos:")
print(datos.dtypes)
print("\nFaltantes por columna:")
print(datos.isna().sum())
print("\nPorcentaje faltante (%):")
print((datos.isna().mean() * 100).round(2))
print("\nDuplicados exactos:", int(datos.duplicated().sum()))
datos.describe().T.round(2)

Tipos de datos:
cemento               float64
escoria               float64
ceniza_volante        float64
agua                  float64
superplastificante    float64
agregado_grueso       float64
agregado_fino         float64
edad_dias               int64
resistencia_mpa       float64
dtype: object

Faltantes por columna:
cemento               0
escoria               0
ceniza_volante        0
agua                  0
superplastificante    0
agregado_grueso       0
agregado_fino         0
edad_dias             0
resistencia_mpa       0
dtype: int64

Porcentaje faltante (%):
cemento               0.0
escoria               0.0
ceniza_volante        0.0
agua                  0.0
superplastificante    0.0
agregado_grueso       0.0
agregado_fino         0.0
edad_dias             0.0
resistencia_mpa       0.0
dtype: float64

Duplicados exactos: 25


,count,mean,std,min,25%,50%,75%,max
cemento,1030.0,281.17,104.51,102.00,192.38,272.90,350.00,540.0
escoria,1030.0,73.90,86.28,0.00,0.00,22.00,142.95,359.4
ceniza_volante,1030.0,54.19,64.00,0.00,0.00,0.00,118.27,200.1
agua,1030.0,181.57,21.36,121.75,164.90,185.00,192.00,247.0
superplastificante,1030.0,6.20,5.97,0.00,0.00,6.35,10.16,32.2
agregado_grueso,1030.0,972.92,77.75,801.00,932.00,968.00,1029.40,1145.0
agregado_fino,1030.0,773.58,80.18,594.00,730.95,779.51,824.00,992.6
edad_dias,1030.0,45.66,63.17,1.00,7.00,28.00,56.00,365.0
resistencia_mpa,1030.0,35.82,16.71,2.33,23.71,34.44,46.14,82.6


## Detección IQR y limpieza conservadora

El criterio IQR marca observaciones fuera de `[Q1 − 1.5·IQR, Q3 + 1.5·IQR]`. Se reportan cuántas hay por variable y se mantienen en el archivo limpio: el criterio estadístico por sí solo no demuestra que sean errores.

In [3]:
filas_iqr = []
for columna in datos.columns:
    q1, q3 = datos[columna].quantile([0.25, 0.75])
    iqr = q3 - q1
    inferior, superior = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    mascara = datos[columna].lt(inferior) | datos[columna].gt(superior)
    filas_iqr.append({"variable": columna, "limite_inferior": inferior,
                      "limite_superior": superior, "n_atipicos": int(mascara.sum())})
resumen_iqr = pd.DataFrame(filas_iqr)
resumen_iqr.round(2)

,variable,limite_inferior,limite_superior,n_atipicos
0,cemento,-44.06,586.44,0
1,escoria,-214.42,357.38,2
2,ceniza_volante,-177.40,295.68,0
3,agua,124.25,232.65,9
4,superplastificante,-15.24,25.40,10
5,agregado_grueso,785.90,1175.50,0
6,agregado_fino,591.38,963.58,5
7,edad_dias,-66.50,129.50,59
8,resistencia_mpa,-9.94,79.78,4


In [4]:
datos_limpios = datos.drop_duplicates().copy()
print("Filas retiradas por duplicación exacta:", len(datos) - len(datos_limpios))
print("Faltantes restantes:", int(datos_limpios.isna().sum().sum()))
salida = Path("Concrete_Data_limpia.csv")
datos_limpios.to_csv(salida, index=False)
print("Guardado en:", salida.name)

Filas retiradas por duplicación exacta: 25
Faltantes restantes: 0
Guardado en: Concrete_Data_limpia.csv


## Conclusión

La base original contiene 1,030 observaciones y 9 variables numéricas. No presenta valores faltantes y tiene 25 filas duplicadas exactas, que se retiran. La revisión IQR identifica algunos valores extremos; se conservan porque el conjunto no documenta que sean errores de medición. El archivo limpio queda con 1,005 observaciones y nombres listos para análisis posteriores.